<a href="https://colab.research.google.com/github/Goseungeun/2022Master/blob/main/Test_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part1

In [ ]:
#Q1
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/subject_performance.csv')

grouped = df.groupby('sub_topic')['is_correct'].mean()
result = grouped.sort_values(ascending=False)
print(result) #  0.674

sub_topic
Algorithms              0.787162
Linear Algebra          0.787162
Mechanics               0.693727
Calculus                0.673913
Quantum                 0.667774
Electromagnetism        0.658621
Operating Systems       0.658621
Discrete Mathematics    0.643836
Databases               0.640523
Name: is_correct, dtype: float64


In [ ]:
#Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/cafe_sales.csv')

# 2-1
df['order_date'] = pd.to_datetime(df['order_date'])
df['year_month'] = df['order_date'].dt.to_period('M')

month_sale = df.groupby('year_month')['price'].sum()
print(month_sale.nlargest(2)) # 328741

# 2-2
top4 = month_sale.nlargest(4)
target_ym = top4.index[-1]
cond = df['year_month'] == target_ym

df = df[cond]
cat_sale = df.groupby('category')['price'].sum()
print(cat_sale.nlargest(1)) # 104660

year_month
2025-03    347848
2025-07    328741
Freq: M, Name: price, dtype: int64
category
smoothie    104660
Name: price, dtype: int64


In [ ]:
#Q3
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/hamspam.csv')

df['words'] = df['text'].str.split()
df['words_num'] = df['words'].str.len()

group = df.groupby('label')['words_num'].mean()
diff = abs(group['spam'] - group['ham'])
print(round(diff,3)) # 0.047

0.047


## Part 2

In [ ]:
import pandas as pd
train = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_test.csv')

#EDA
print(train.shape)
print(test.shape)

print(train.info())

print(train.isnull().sum().sum())
print(test.isnull().sum().sum())

print(train['시군구명'].nunique())
print(test['시군구명'].nunique())

#Preprocessing
target = train.pop('총가스사용량')
train = pd.get_dummies(train)
test = pd.get_dummies(test)

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

X_train,X_val,y_train,y_val = train_test_split(train,target,test_size = 0.2,random_state=0)
rf = RandomForestRegressor()
rf.fit(X_train,y_train)
val_pred = rf.predict(X_val)

pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv('result.csv',index=False)
print(pd.read_csv('result.csv').head())
print(result.shape)

(3196, 6)
(1476, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3196 entries, 0 to 3195
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   시군구명    3196 non-null   object 
 1   생활및판매   3196 non-null   int64  
 2   공공문화    3196 non-null   int64  
 3   복지의료    3196 non-null   int64  
 4   업무오락체육  3196 non-null   int64  
 5   총가스사용량  3196 non-null   float64
dtypes: float64(1), int64(4), object(1)
memory usage: 149.9+ KB
None
0
0
21
21
           pred
0   9841.640285
1  10766.798380
2   9111.151794
3   9841.640285
4  11219.917610
(1476, 1)


## Part3

In [53]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/attrition.csv')

from statsmodels.formula.api import logit
import numpy as np

#1-1
model = logit('attrition~age+income+overtime',data=df).fit()
print(model.summary()) # p_value <0.05 => income
print(round(model.params['income'],3)) #-0.002

# 1-2
print(np.exp(model.params['age'])) #0.996

# 1-3
new_data = pd.DataFrame({'age' : [40], 'income' :[4500], 'overtime':[1]})
pred = model.predict(new_data)
print(pred) # 0.697

Optimization terminated successfully.
         Current function value: 0.387298
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:              attrition   No. Observations:                 6000
Model:                          Logit   Df Residuals:                     5996
Method:                           MLE   Df Model:                            3
Date:                Thu, 18 Jun 2026   Pseudo R-squ.:                  0.4215
Time:                        05:41:47   Log-Likelihood:                -2323.8
converged:                       True   LL-Null:                       -4016.9
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      7.7724      0.246     31.650      0.000       7.291       8.254
age           -0.0044      0.

In [62]:
# Q2
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/heating.csv')

from statsmodels.formula.api import ols
model = ols('heating_load ~ wall + roof+glazing+height',data = df).fit()
print(model.summary())
# 2-1
print(model.params[1:].sum()) # 0.254

# 2-2
model2 = ols('heating_load ~ roof+glazing+height',data = df).fit()
print(round(model2.rsquared,3)) # 0.754

#2-3
new = pd.DataFrame({'wall':[20],'roof':[150],'glazing':[20],'height':[5]})
pred = model2.predict(new)
print(round(pred[0],3)) # 79.612


                            OLS Regression Results                            
Dep. Variable:           heating_load   R-squared:                       0.754
Model:                            OLS   Adj. R-squared:                  0.752
Method:                 Least Squares   F-statistic:                     417.8
Date:                Thu, 18 Jun 2026   Prob (F-statistic):          2.02e-164
Time:                        05:59:22   Log-Likelihood:                -1772.0
No. Observations:                 550   AIC:                             3554.
Df Residuals:                     545   BIC:                             3576.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     38.3821      1.504     25.517      0.0